In [1]:
USING_KAGGLE = True
import os
import math
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/keysss/kaggle_deploying
/kaggle/input/keysss/kaggle_deploying.pub


In [2]:
#%pip install uv -q
!mkdir best_run_plots tuned_run_plots default_run_plots varying_rs_run_plots models models/tuning models/default models/tuned models/varying_random_state
%pip install pyngrok mlflow lion_pytorch adabound torch-optimizer optax onnx onnxscript ray[tune] adabelief-pytorch==0.2.0 -q
%pip install --upgrade seaborn -q
%pip install protobuf==3.20.0 -q

In [3]:
!sudo chmod 600 /kaggle/working/key
!ssh-keyscan -t ecdsa github.com
!GIT_SSH_COMMAND='sudo ssh -i /kaggle/working/key -o IdentitiesOnly=yes -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null' git clone git@github.com:maximspbu/optimizers_comparison.git
!cd optimizers_comparison && git switch sprint5

# github.com:22 SSH-2.0-0264bb16
github.com ecdsa-sha2-nistp256 AAAAE2VjZHNhLXNoYTItbmlzdHAyNTYAAAAIbmlzdHAyNTYAAABBBEmKSENjQEezOmxkZMy7opKgwFB9nkt5YRrYMjNuG5N87uRgg6CLrbo5wAdT/y6v0mKV0U2w0WZ2YB/++Tpockg=
Cloning into 'optimizers_comparison'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 103 (delta 46), reused 77 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 5.47 MiB | 20.91 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Branch 'sprint5' set up to track remote branch 'sprint5' from 'origin'.
Switched to a new branch 'sprint5'


In [4]:
%pip install lion_pytorch torch-optimizer ray[tune] adabelief-pytorch==0.2.0 -q
%pip install --upgrade seaborn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [ ]:
from copy import copy
from typing import Optional
import time
import inspect

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchmetrics import MetricCollection
from torchmetrics.regression import MeanAbsolutePercentageError as MAPE, R2Score
from torchvision import transforms
from torchvision.transforms import v2, ToTensor
from torchinfo import summary

import pytorch_lightning as pl
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning import seed_everything

import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

import random

import seaborn as sns

import ray
from ray.train.torch import enable_reproducibility
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
from ray.air.integrations.mlflow import MLflowLoggerCallback, setup_mlflow
from ray.tune import JupyterNotebookReporter
from ray.tune.schedulers import ASHAScheduler
from ray.air import session

from pprint import pprint

from torch.optim import AdamW
from torch_optimizer import Adahessian as AdaHessian
from adabelief_pytorch import AdaBelief
from lion_pytorch import Lion

import matplotlib.pyplot as plt

In [ ]:
from torch.optim import Optimizer


class AdaBound(Optimizer):
    """Implements AdaBound algorithm.
    It has been proposed in `Adaptive Gradient Methods with Dynamic Bound of Learning Rate`_.
    Arguments:
        params (iterable): iterable of parameters to optimize or dicts defining
            parameter groups
        lr (float, optional): Adam learning rate (default: 1e-3)
        betas (Tuple[float, float], optional): coefficients used for computing
            running averages of gradient and its square (default: (0.9, 0.999))
        final_lr (float, optional): final (SGD) learning rate (default: 0.1)
        gamma (float, optional): convergence speed of the bound functions (default: 1e-3)
        eps (float, optional): term added to the denominator to improve
            numerical stability (default: 1e-8)
        weight_decay (float, optional): weight decay (L2 penalty) (default: 0)
        amsbound (boolean, optional): whether to use the AMSBound variant of this algorithm
    .. Adaptive Gradient Methods with Dynamic Bound of Learning Rate:
        https://openreview.net/forum?id=Bkg3g2R9FX
    """

    def __init__(
        self,
        params,
        lr=1e-3,
        betas=(0.9, 0.999),
        final_lr=0.1,
        gamma=1e-3,
        eps=1e-8,
        weight_decay=0,
        amsbound=False,
    ):
        if not 0.0 <= lr:
            raise ValueError("Invalid learning rate: {}".format(lr))
        if not 0.0 <= eps:
            raise ValueError("Invalid epsilon value: {}".format(eps))
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError("Invalid beta parameter at index 0: {}".format(betas[0]))
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError("Invalid beta parameter at index 1: {}".format(betas[1]))
        if not 0.0 <= final_lr:
            raise ValueError("Invalid final learning rate: {}".format(final_lr))
        if not 0.0 <= gamma < 1.0:
            raise ValueError("Invalid gamma parameter: {}".format(gamma))
        defaults = dict(
            lr=lr,
            betas=betas,
            final_lr=final_lr,
            gamma=gamma,
            eps=eps,
            weight_decay=weight_decay,
            amsbound=amsbound,
        )
        super(AdaBound, self).__init__(params, defaults)

        self.base_lrs = list(map(lambda group: group["lr"], self.param_groups))

    def __setstate__(self, state):
        super(AdaBound, self).__setstate__(state)
        for group in self.param_groups:
            group.setdefault("amsbound", False)

    def step(self, closure=None):
        """Performs a single optimization step.
        Arguments:
            closure (callable, optional): A closure that reevaluates the model
                and returns the loss.
        """
        loss = 0
        if closure is not None:
            loss = closure()

        for group, base_lr in zip(self.param_groups, self.base_lrs):
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError(
                        "Adam does not support sparse gradients, please consider SparseAdam instead"
                    )
                amsbound = group["amsbound"]

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state["step"] = 0
                    # Exponential moving average of gradient values
                    state["exp_avg"] = torch.zeros_like(p.data)
                    # Exponential moving average of squared gradient values
                    state["exp_avg_sq"] = torch.zeros_like(p.data)
                    if amsbound:
                        # Maintains max of all exp. moving avg. of sq. grad. values
                        state["max_exp_avg_sq"] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                if amsbound:
                    max_exp_avg_sq = state["max_exp_avg_sq"]
                beta1, beta2 = group["betas"]

                state["step"] += 1

                if group["weight_decay"] != 0:
                    # grad = grad.add(group['weight_decay'], p.data)
                    grad = grad.add(p.data, alpha=group["weight_decay"])

                # Decay the first and second moment running average coefficient
                # exp_avg.mul_(beta1).add_(1 - beta1, grad)
                # exp_avg_sq.mul_(beta2).addcmul_(1 - beta2, grad, grad)
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                if amsbound:
                    # Maintains the maximum of all 2nd moment running avg. till now
                    torch.max(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    # Use the max. for normalizing running avg. of gradient
                    denom = max_exp_avg_sq.sqrt().add_(group["eps"])
                else:
                    denom = exp_avg_sq.sqrt().add_(group["eps"])

                bias_correction1 = 1 - beta1 ** state["step"]
                bias_correction2 = 1 - beta2 ** state["step"]
                step_size = group["lr"] * math.sqrt(bias_correction2) / bias_correction1

                # Applies bounds on actual learning rate
                # lr_scheduler cannot affect final_lr, this is a workaround to apply lr decay
                final_lr = group["final_lr"] * group["lr"] / base_lr
                lower_bound = final_lr * (1 - 1 / (group["gamma"] * state["step"] + 1))
                upper_bound = final_lr * (1 + 1 / (group["gamma"] * state["step"]))
                step_size = torch.full_like(denom, step_size)
                step_size.div_(denom).clamp_(lower_bound, upper_bound).mul_(exp_avg)

                p.data.add_(-step_size)

        return loss


2025-05-13 07:10:12.046045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747120212.267794      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747120212.327871      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
def preprocess_data(filename: str) -> torch.Tensor:
    """Function to preprocess data

    Args:
        filename (str): Filename of data to be preprocessed.

    Returns:
        torch.Tensor: preprocessed data
    """
    data = pd.read_csv(filename)

    data_t = torch.Tensor(data.values)
    data_t = torch.nn.functional.normalize(data_t)

    return data_t


def get_train_valid_test_datasets(
    train_filename: str, test_filename: str
) -> Tuple[TensorDataset, ...]:
    train_df = pd.read_csv(train_filename)
    test_df = pd.read_csv(test_filename)

    X_train, X_valid, y_train, y_valid = train_test_split(
        train_df.iloc[:, :-1].values, train_df.iloc[:, -1].values, random_state=0
    )

    X_train_np = X_train.astype(np.float32)
    y_train_np = y_train.astype(np.float32).reshape(-1, 1)
    X_valid_np = X_valid.astype(np.float32)
    y_valid_np = y_valid.astype(np.float32).reshape(-1, 1)

    X_test_np = test_df.iloc[:, :-1].values.astype(np.float32)
    y_test_np = test_df.iloc[:, -1].values.astype(np.float32).reshape(-1, 1)

    scaler = StandardScaler()
    X_train_scaled_np = scaler.fit_transform(X_train_np)
    X_valid_scaled_np = scaler.fit_transform(X_valid_np)
    X_test_scaled_np = scaler.transform(X_test_np)

    X_train_tensor = torch.tensor(X_train_scaled_np, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)
    X_valid_tensor = torch.tensor(X_valid_scaled_np, dtype=torch.float32)
    y_valid_tensor = torch.tensor(y_valid_np, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_scaled_np, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    valid_dataset = TensorDataset(X_valid_tensor, y_valid_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    return train_dataset, valid_dataset, test_dataset


2025-05-13 07:10:27,276	INFO worker.py:1852 -- Started a local Ray instance.
2025-05-13 07:10:27,347	INFO packaging.py:575 -- Creating a file package for local module '.'.
2025-05-13 07:10:27,426	INFO packaging.py:367 -- Pushing file package 'gcs://_ray_pkg_31c25b91c219b3a0.zip' (12.10MiB) to Ray cluster...
2025-05-13 07:10:27,511	INFO packaging.py:380 -- Successfully pushed file package 'gcs://_ray_pkg_31c25b91c219b3a0.zip'.


{'CPU': 4.0,
 'GPU': 2.0,
 'accelerator_type:T4': 1.0,
 'memory': 20685609370.0,
 'node:172.19.2.2': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 8865261158.0}


In [8]:
train_filename = "/kaggle/input/d/panosc/california-housing-prices/S7_california_housing_train.csv"
test_filename = "/kaggle/input/d/panosc/california-housing-prices/S7_california_housing_test.csv"

train_dataset, valid_dataset, test_dataset = get_train_valid_test_datasets(
    train_filename=train_filename, test_filename=test_filename
)


In [ ]:
SEED = 0
NUM_THREADS = 4
def setup(working_dir: str = ".") -> None:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    pd.set_option("display.max_colwidth", None)
    ray.shutdown()
    ray.init(
       runtime_env={
           "env_vars": {"RAY_AIR_RICH_LAYOUT": "1"},
           "working_dir": working_dir,
       },
    )
    resources = ray.cluster_resources()
    pprint(resources)
    torch.set_num_threads(NUM_THREADS)

def seed_worker(worker_id: int):
    np.random.seed(SEED)
    random.seed(SEED)


def get_or_create_experiment(experiment_name):
    """
    Retrieve the ID of an existing MLflow experiment or create a new one if it doesn't exist.
    
    This function checks if an experiment with the given name exists within MLflow.
    If it does, the function returns its ID. If not, it creates a new experiment
    with the provided name and returns its ID.
    
    Parameters:
    - experiment_name (str): Name of the MLflow experiment.
    
    Returns:
    - str: ID of the existing or newly created MLflow experiment.
    """
    
    if experiment := mlflow.get_experiment_by_name(experiment_name):
        return experiment.experiment_id
    return mlflow.create_experiment(experiment_name)

from pyngrok import ngrok
from getpass import getpass

ngrok.kill()
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

setup(".")
experiment_name = "regression"
#mlflow.set_tracking_uri(tracking_uri)
experiment_id = get_or_create_experiment("regression")

mlflow.set_tracking_uri("./mlruns") 
mlflow.set_experiment(experiment_name)

get_ipython().system_raw("mlflow ui --backend-store-uri ./mlruns --host 0.0.0.0 &")
public_url = ngrok.connect(5000, "http", host_header="localhost:5000")
tracking_uri = "./mlruns"
client = MlflowClient()
print(f"✅ MLflow UI is live at: {public_url}")

In [ ]:
class SimpleRegressionModel(nn.Module):
    def __init__(
        self,
        input_shape: int = 8,
        hidden_units: int = 64,
        output_shape: int = 1,
        activation_function: nn.Module = nn.ReLU(),
        normalization: nn.Module | None = None,
        dropout: nn.Module | None = None,
        device: torch.device = DEVICE,
    ) -> None:
        """SimpleRegressionModel initializer

        Args:
            input_shape (int): Number of units in input layer
            hidden_units (int): Number of units in hidden layers
            output_shape (int): Number of units in output layer
            activation_function (Optional[torch.nn.Module], optional): Activation function. Defaults to None.
            normalization (Optional[torch.nn.Module], optional): Use batch or layer normalization. Defaults to None.
            dropout (Optional[torch.nn.Module], optional): Dropout layer. Defaults to None.
        """
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(
                in_features=input_shape,
                out_features=hidden_units,
                device=device,
            ),
            nn.Identity() if normalization is None else normalization,
            nn.Identity() if activation_function is None else activation_function,
            nn.Identity() if dropout is None else dropout,
            nn.Linear(
                in_features=hidden_units,
                out_features=hidden_units,
                device=device,
            ),
            nn.Identity() if activation_function is None else activation_function,
            nn.Linear(
                in_features=hidden_units,
                out_features=output_shape,
                device=device,
            ),
            # nn.Flatten(),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """Method for forward pass.

        Args:
            X (torch.Tensor): Input data.

        Returns:
            torch.Tensor: Result of computations.
        """
        return self.block(X)



In [10]:
INPUT_SHAPE = 54
HIDDEN_UNITS = 54
summary(SimpleRegressionModel(input_shape=INPUT_SHAPE, hidden_units=HIDDEN_UNITS,))

Layer (type:depth-idx)                   Param #
SimpleRegressionModel                    --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       576
│    └─Identity: 2-2                     --
│    └─ReLU: 2-3                         --
│    └─Identity: 2-4                     --
│    └─Linear: 2-5                       4,160
│    └─ReLU: 2-6                         --
│    └─Linear: 2-7                       65
Total params: 4,801
Trainable params: 4,801
Non-trainable params: 0

In [ ]:
NUM_EPOCHS = 31
EPOCH_STEP = 1
BATCH_SIZE = 128

OPTIMIZERS_PARAMS: dict = {
    AdaBelief: {
        "lr": tune.grid_search([1e-4, 1e-3, 1e-2, 1e-1]),
        # "weight_decay": tune.grid_search([0, 1e-4, 1e-3, 1e-2]),
        # "amsgrad": tune.grid_search([False, True]),
        # "weight_decouple": tune.grid_search([False, True]),
        # "rectify": tune.grid_search([False, True]),
        "print_change_log": tune.grid_search([False]),
    },
    # AdaHessian: {
    #     "lr": tune.grid_search([1e-2, 1e-1, 0.15, 1.0]),
    #     "weight_decay": tune.grid_search([0, 1e-4, 1e-3, 1e-2]),
    #     "hessian_power": tune.grid_search([0.5, 0.75, 1.0]),
    #     "seed": tune.grid_search([SEED]),
    # },
    Lion: {
        "lr": tune.grid_search([1e-4, 1e-3, 1e-2, 1e-1]),
        # "weight_decay": tune.grid_search([1e-4, 1e-3, 1e-2]),
        # "decoupled_weight_decay": tune.grid_search([False, True]),
        "use_triton": tune.grid_search([False]),
    },
    # AdamW: {
    #     "lr": tune.grid_search([1e-4, 1e-3, 1e-2, 1e-1]),
    #     "weight_decay": tune.grid_search([0, 1e-4, 1e-3, 1e-2]),
    #     "amsgrad": tune.grid_search([False, True]),
    # },
    # AdaBound: {
    #     "lr": tune.grid_search([1e-5, 1e-4, 1e-3, 1e-2]),
    #     "final_lr": tune.grid_search([1e-4, 1e-3, 1e-2, 1e-1]),
    #     "gamma": tune.grid_search([1e-4, 1e-3, 1e-2]),
    #     "weight_decay": tune.grid_search([0, 1e-4, 1e-3]),
    #     "amsbound": tune.grid_search([False, True]),
    # },
}

analysis_optimizer = {}
metrics = {
    "mape": MAPE(), 
    "r2score": R2Score(),}

reporter = JupyterNotebookReporter(
    metric_columns=["vsl_loss"], 
    mode="min", 
    print_intermediate_tables=False,
)

train_dataset_ref = ray.put(train_dataset)
valid_dataset_ref = ray.put(valid_dataset)
test_dataset_ref = ray.put(test_dataset)

mlflow_callback = MLflowLoggerCallback(
    tracking_uri="./mlruns",
    experiment_name="mnist_ray_tune_experiment",
    save_artifact=True,
)

for metric in metrics:
    reporter.add_metric_column(f"valid_{metric}")

for optimizer_type in OPTIMIZERS_PARAMS.keys():
    def wrap(config: dict) -> None:
        """Wrapper function for tuning hyperparameters.

        Args:
            config: A dict whose keys are parameters to be tuned and values that the parameter can take.

        """
        setup_mlflow(
            config,
            experiment_name=config.get("experiment_name", None),
            tracking_uri=config.get("tracking_uri", None),
        )
        train_dataloader = DataLoader(
            dataset=ray.get(train_dataset_ref),
            batch_size=BATCH_SIZE,
            pin_memory=True,
            num_workers=2,
            shuffle=True,
            worker_init_fn=seed_worker,
            generator=TORCH_GENERATOR,
        )
        valid_dataloader = DataLoader(
            dataset=ray.get(valid_dataset_ref),
            batch_size=BATCH_SIZE,
            num_workers=2,
            pin_memory=True,
            shuffle=False,
            worker_init_fn=seed_worker,
            generator=TORCH_GENERATOR,
        )
        test_dataloader = DataLoader(
            dataset=ray.get(test_dataset_ref),
            batch_size=BATCH_SIZE,
            num_workers=2,
            pin_memory=True,
            shuffle=False,
            worker_init_fn=seed_worker,
            generator=TORCH_GENERATOR,
        )
        model = SimpleRegressionModel(input_shape=INPUT_SHAPE, 
                                      hidden_units=HIDDEN_UNITS, 
                                      activation_function=nn.ReLU(),)
        loss_fn = torch.nn.MSELoss()
        wrapper = LightningWrapper(model=model, 
                                   optimizer_class=optimizer_type, 
                                   optimizer_hparams={i: j for i, j in config.items() if i not in ["experiment_name", "tracking_uri"]}, 
                                   loss_fn=loss_fn, )
        tune_callback = TuneReportCallback(
            {
                "val_loss": "val_loss",
                "val_mape": "val_mape",
                "val_r2score": "val_r2score",
            },
            on="validation_end"
        )
        trainer = pl.Trainer(callbacks=[tune_callback, ],
                            limit_train_batches=BATCH_SIZE, 
                            max_epochs=NUM_EPOCHS,
                            accelerator="auto",
                            devices="auto",
                            enable_progress_bar=False,
                            enable_checkpointing=False,
        )
        
        trainer.fit(model=wrapper, 
                    train_dataloaders=train_dataloader,
                    val_dataloaders=valid_dataloader,)
        model.eval() 
    
        dummy_input = torch.randn(1, 3, 32, INPUT_SHAPE) 
    
        trial_name = session.get_trial_name()
    
        optimizer_name = str(optimizer_type).split('.')[-1].replace("'>", "")
    
        output_dir = "/kaggle/working/models/tuning"
        os.makedirs(output_dir, exist_ok=True)
        
        onnx_file_path = os.path.join(output_dir, f"{optimizer_name}_{trial_name}.onnx")
        torch.onnx.export(
            model,
            dummy_input,
            onnx_file_path,
            export_params=True,
            opset_version=11, # A common opset version
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={'input' : {0 : 'batch_size'},
                          'output' : {0 : 'batch_size'}}
        )
        # print(f"Successfully exported ONNX model to: {onnx_file_path}")
        
    analysis = tune.run(
        wrap,
        config={str(key): value for key, value in OPTIMIZERS_PARAMS[optimizer_type].items()} | 
        {"experiment_name": experiment_name, "tracking_uri": tracking_uri, },
        scheduler=ASHAScheduler(metric='val_accuracy', mode='max'),
        progress_reporter=reporter,
        resources_per_trial={"cpu": 2, 
                             "gpu": 1,
                             },
        log_to_file=True,
        callbacks=[mlflow_callback],
    )
    analysis_optimizer[optimizer_type] = analysis


In [ ]:
def get_mlflow_run_id_from_trial(experiment_name: str, trial) -> str | None:
    """
    Finds the MLflow run ID by searching for a run with a 'trial_name' tag
    that matches the string representation of the Ray Tune Trial.

    Args:
        experiment_name: The name of the MLflow experiment.
        trial: The Ray Tune Trial object.

    Returns:
        The MLflow run ID as a string, or None if not found.
    """
    try:
        experiment = client.get_experiment_by_name(experiment_name)
        if not experiment:
            print(f"Error: Experiment '{experiment_name}' not found.")
            return None
        experiment_id = experiment.experiment_id

        filter_string = f"tags.`trial_name` = '{str(trial)}'"
        runs = client.search_runs(experiment_ids=[experiment_id], filter_string=filter_string)
        
        if not runs:
            print(f"Warning: No MLflow run found for trial '{trial}'")
            return None
        
        return runs[0].info.run_id

    except Exception as e:
        print(f"An error occurred while searching for MLflow run for trial {trial}: {e}")
        return None

def get_metric_history_df(run_id: str, metric_name: str) -> pd.DataFrame:
    """Fetches the full history of a metric for a given run and returns it as a DataFrame."""
    history = client.get_metric_history(run_id, metric_name)
    
    df = pd.DataFrame(
        [{'step': m.step, 'value': m.value, 'timestamp': m.timestamp} for m in history]
    )
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df
    
experiment_name = 'regression'
val_metric_dfs = {}
for opt_cls in OPTIMIZERS_PARAMS:
    analysis = cls_analysis_optimizer[opt_cls]
    best_trial = analysis.get_best_trial(metric=f"val_accuracy", mode="max")
    mlflow_run_id = get_mlflow_run_id_from_trial(experiment_name, best_trial)
    val_metric_dfs[opt_cls] = {}
    val_metric_dfs[opt_cls]['loss'] = get_metric_history_df(mlflow_run_id, "val_loss")
    val_metric_dfs[opt_cls]['mape'] = get_metric_history_df(mlflow_run_id, "val_mape")
    val_metric_dfs[opt_cls]['r2score'] = get_metric_history_df(mlflow_run_id, "val_r2score")
    print(f"Running for {opt_cls=}")
    print("\nValidation Loss History:")
    print(val_metric_dfs[opt_cls]['loss'].head())

In [ ]:
runs = mlflow.search_runs()
runs.sort_values('start_time', ascending=False)

In [ ]:
sns.set_style("whitegrid")

for metric in val_metric_dfs[next(iter(val_metric_dfs))]:
    for opt_cls in val_metric_dfs:
        plt.plot(val_metric_dfs[opt_cls][metric]['step'][:-1], val_metric_dfs[opt_cls][metric]['value'][:-1], label=opt_cls)
    plt.title(f'Validation {metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    output_filename = f"best_run_plots/best_run_{metric}.png"
    plt.savefig(output_filename, dpi=300)
    print(f"\nPlot saved to {output_filename}")
    plt.show()

In [12]:
optimizer_comparison_mape_df = pd.DataFrame({"Optimizer": [repr(i)[repr(i).rfind('.') + 1: -2] for i in cls_analysis_optimizer], 
                                            "Config": [i.get_best_config(metric=f"val_r2score", mode="max") for i in cls_analysis_optimizer.values()], 
                                            "MAPE": [i.get_best_trial(metric=f"val_mape", mode="max").last_result[f"val_mape"] for i in cls_analysis_optimizer.values() ], 
                                            "R2Score": [i.get_best_trial(metric=f"val_accuracy", mode="max").last_result[f"val_r2score"] for i in cls_analysis_optimizer.values()], 
                                            "Time_total_s": [round(i.get_best_trial(metric=f"val_accuracy", mode="max").last_result['time_total_s'], 2) for i in cls_analysis_optimizer.values()],
                                           })

optimizer_comparison_mape_df

,Optimizer,Config,MAPE,Time_total_s
0,Adahessian,"{'lr': 0.1, 'weight_decay': 0, 'hessian_power': 1.0, 'seed': 0}",0.206933,46.053466
1,Lion,"{'lr': 0.1, 'weight_decay': 0.0001, 'decoupled_weight_decay': True, 'use_triton': False}",0.217938,31.553410
2,AdaBelief,"{'lr': 0.1, 'weight_decay': 0.01, 'amsgrad': True, 'weight_decouple': True, 'rectify': False, 'print_change_log': False}",0.247575,34.516403
3,AdamW,"{'lr': 0.1, 'weight_decay': 0.01, 'amsgrad': False}",0.247539,31.026601
4,AdaBound,"{'lr': 1e-05, 'final_lr': 1e-07, 'gamma': 0.01, 'weight_decay': 0.001, 'amsbound': False}",0.207838,35.023025


In [13]:
results = {}
TEST_NUM_EPOCHS = 51
TEST_EPOCH_STEP = 1

train_dataloader = DataLoader(
    dataset=ray.get(train_dataset_ref),
    batch_size=BATCH_SIZE,
    pin_memory=True,
    num_workers=2,
    shuffle=True,
    worker_init_fn=seed_worker,
    generator=TORCH_GENERATOR,
)
valid_dataloader = DataLoader(
    dataset=ray.get(valid_dataset_ref),
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=TORCH_GENERATOR,
)
test_dataloader = DataLoader(
    dataset=ray.get(test_dataset_ref),
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True,
    shuffle=False,
    worker_init_fn=seed_worker,
    generator=TORCH_GENERATOR,
)

experiment_name = "tuned_regression"
experiment_id = get_or_create_experiment(experiment_name)
mlflow.set_experiment(experiment_id=experiment_id)

for optimizer_type, config in zip(OPTIMIZERS_PARAMS.keys(), optimizer_comparison_acc_df["Config"]):
    with mlflow.start_run(run_name=experiment_name+f"_opt_{str(optimizer_type)[str(optimizer_type).rfind('.')+1:-2]}") as run:
        model = SimpleRegressionModel(input_shape=INPUT_SHAPE, 
                                      hidden_units=HIDDEN_UNITS, 
                                      activation_function=nn.ReLU(),
                                     )
        loss_fn = nn.CrossEntropyLoss()
        wrapper = LightningWrapper(model=model, 
                                   optimizer_class=optimizer_type, 
                                   optimizer_hparams={i: j for i, j in config.items() if i not in ["experiment_name", "tracking_uri"]}, 
                                   loss_fn=loss_fn, )
        tune_callback = TuneReportCallback(
            {
                "val_loss": "val_loss",
                "val_mape": "val_mape",
                "val_r2score": "val_r2score",
            },
            on="validation_end"
        )
        mlflow.pytorch.autolog()
        trainer = pl.Trainer(#callbacks=[tune_callback, ],
                            limit_train_batches=BATCH_SIZE, 
                            max_epochs=NUM_EPOCHS,
                            accelerator="auto",
                            devices="auto",
                            enable_progress_bar=False,
                            enable_checkpointing=False,
        )
        start = time.perf_counter()
        trainer.fit(model=wrapper, 
                    train_dataloaders=train_dataloader,
                    val_dataloaders=valid_dataloader,)
        end = time.perf_counter()
        time_training = end - start
        metrics_to_save = trainer.test(model=wrapper, dataloaders=test_dataloader)[0]
        # print(metrics_to_save)
        metrics_to_save = {
            "test_mape": metrics_to_save.get("test_mape", 0.),
            "test_r2score": metrics_to_save.get("test_r2score", 0.),
        }
        metrics_to_save['time_training'] = time_training
        optimizer_name = str(optimizer_type)[str(optimizer_type).rfind('.')+1:-2]
        print(f"{optimizer_name} was trained for {time_training} seconds")
        results[optimizer_name] = metrics_to_save
        
        model.eval() 
    
        dummy_input = torch.randn(1, 3, 32, INPUT_SHAPE) 
    
        trial_name = session.get_trial_name()
    
        output_dir = "/kaggle/working/models/tuned"
        os.makedirs(output_dir, exist_ok=True)
        
        onnx_file_path = os.path.join(output_dir, f"{optimizer_name}.onnx")
        torch.onnx.export(
            model,
            dummy_input,
            onnx_file_path,
            export_params=True,
            opset_version=11,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={'input' : {0 : 'batch_size'},
                          'output' : {0 : 'batch_size'}}
        )

,Optimizer,Config,R2Score,Time_total_s
0,Adahessian,"{'lr': 0.01, 'weight_decay': 0.001, 'hessian_power': 0.75, 'seed': 0}",0.756148,46.411203
1,Lion,"{'lr': 0.1, 'weight_decay': 0.0001, 'decoupled_weight_decay': True, 'use_triton': False}",0.747879,31.553410
2,AdaBelief,"{'lr': 0.1, 'weight_decay': 0.001, 'amsgrad': False, 'weight_decouple': True, 'rectify': False, 'print_change_log': False}",0.706242,33.956667
3,AdamW,"{'lr': 0.1, 'weight_decay': 0.001, 'amsgrad': False}",0.704664,30.978518
4,AdaBound,"{'lr': 0.0001, 'final_lr': 1e-07, 'gamma': 0.01, 'weight_decay': 0, 'amsbound': False}",0.729952,33.727226


In [ ]:
def get_metric_history_df(run_id: str, metric_name: str) -> pd.DataFrame:
    """Fetches the full history of a metric for a given run and returns it as a DataFrame."""
    history = client.get_metric_history(run_id, metric_name)
    #print(history)
    df = pd.DataFrame(
        [{'step': m.step, 'value': m.value, 'timestamp': m.timestamp} for m in history]
    )
    #print(df)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df

experiment_name = "tuned_regression"
tuned_val_metric_dfs = {}
for opt_cls in results:
    analysis = results[opt_cls]
    mlflow_run_id = mlflow.search_runs(experiment_ids=[experiment_id], filter_string=f"attributes.run_name = '{experiment_name + '_opt_' + opt_cls}'").sort_values(by='start_time', ascending=False).iloc[0]['run_id']
    print(mlflow_run_id)
    tuned_val_metric_dfs[opt_cls] = {}
    tuned_val_metric_dfs[opt_cls]['loss'] = get_metric_history_df(mlflow_run_id, "val_loss")
    tuned_val_metric_dfs[opt_cls]['mape'] = get_metric_history_df(mlflow_run_id, "val_mape")
    tuned_val_metric_dfs[opt_cls]['r2score'] = get_metric_history_df(mlflow_run_id, "val_r2score")
    print(f"Running for {opt_cls=}")
    print("\nValidation Loss History:")
    print(tuned_val_metric_dfs[opt_cls]['loss'].head())

In [ ]:
for metric in tuned_val_metric_dfs[next(iter(tuned_val_metric_dfs))]:
    for opt_cls in tuned_val_metric_dfs:
        plt.plot(tuned_val_metric_dfs[opt_cls][metric]['step'][:-1], tuned_val_metric_dfs[opt_cls][metric]['value'][:-1], label=opt_cls)
    plt.title(f'Tuned: Validation {metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    output_filename = f"tuned_run_plots/best_tuned_run_{metric}.png"
    plt.savefig(output_filename, dpi=300)
    print(f"\nPlot saved to {output_filename}")
    plt.show()

In [ ]:
default_results = {}

experiment_name = "default_regression"
experiment_id = get_or_create_experiment(experiment_name)
mlflow.set_experiment(experiment_id=experiment_id)

for optimizer_type in OPTIMIZERS_PARAMS.keys():
    with mlflow.start_run(run_name=experiment_name+f"_opt_{str(optimizer_type)[str(optimizer_type).rfind('.')+1:-2]}") as run:
        model = SimpleRegressionModel(input_shape=INPUT_SHAPE, 
                                      hidden_units=HIDDEN_UNITS, 
                                      activation_function=nn.ReLU(),
                                     )
        loss_fn = nn.CrossEntropyLoss()
        wrapper = LightningWrapper(model=model, 
                                   optimizer_class=optimizer_type, 
                                   optimizer_hparams={}, 
                                   loss_fn=loss_fn, )
        mlflow.pytorch.autolog()
        trainer = pl.Trainer(#callbacks=[tune_callback, ],
                            limit_train_batches=BATCH_SIZE, 
                            max_epochs=NUM_EPOCHS,
                            accelerator="auto",
                            devices="auto",
                            enable_progress_bar=False,
                            #strategy="ddp_notebook",
                            enable_checkpointing=False,
                            #logger=False,
        )
        start = time.perf_counter()
        trainer.fit(model=wrapper, 
                    train_dataloaders=train_dataloader,
                    val_dataloaders=valid_dataloader,)
        end = time.perf_counter()
        time_training = end - start
        metrics_to_save = trainer.test(model=wrapper, dataloaders=test_dataloader)[0]
        metrics_to_save = {
            "test_mape": metrics_to_save.get("test_mape", 0.),
            "test_r2score": metrics_to_save.get("test_r2score", 0.),
        }
        metrics_to_save['time_training'] = time_training
        optimizer_name = 'default_' + str(optimizer_type)[str(optimizer_type).rfind('.')+1:-2]
        print(f"{optimizer_name} was trained for {time_training} seconds")
        default_results[optimizer_name] = metrics_to_save
        
        model.eval() 
    
        dummy_input = torch.randn(1, 3, 32, INPUT_SHAPE) 
    
        trial_name = session.get_trial_name()
    
        optimizer_name = str(optimizer_type).split('.')[-1].replace("'>", "")
    
        output_dir = "/kaggle/working/models/default"
        os.makedirs(output_dir, exist_ok=True)
        
        onnx_file_path = os.path.join(output_dir, f"{optimizer_name}.onnx")
        torch.onnx.export(
            model,
            dummy_input,
            onnx_file_path,
            export_params=True,
            opset_version=11, # A common opset version
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={'input' : {0 : 'batch_size'},
                          'output' : {0 : 'batch_size'}}
        )
        print(f"Successfully exported ONNX model to: {onnx_file_path}")
        

In [ ]:
default_val_metric_dfs = {}
experiment_name = "default_classification"
experiment_id = get_or_create_experiment(experiment_name)
for opt_cls in default_results:
    analysis = default_results[opt_cls]
    mlflow_run_id = mlflow.search_runs(experiment_ids=[experiment_id], filter_string=f"attributes.run_name = '{experiment_name}_opt_{opt_cls[opt_cls.find('_') + 1:]}'").sort_values(by='start_time', ascending=False).iloc[0]['run_id']
    default_val_metric_dfs[opt_cls] = {}
    default_val_metric_dfs[opt_cls]['loss'] = get_metric_history_df(mlflow_run_id, "val_loss")
    default_val_metric_dfs[opt_cls]['accuracy'] = get_metric_history_df(mlflow_run_id, "val_accuracy")
    default_val_metric_dfs[opt_cls]['f1score'] = get_metric_history_df(mlflow_run_id, "val_f1score")
    default_val_metric_dfs[opt_cls]['precision'] = get_metric_history_df(mlflow_run_id, "val_precision")
    default_val_metric_dfs[opt_cls]['recall'] = get_metric_history_df(mlflow_run_id, "val_recall")
    print(f"Running for {opt_cls=}")
    print("\nValidation Loss History:")
    print(default_val_metric_dfs[opt_cls]['loss'].head())

In [ ]:
for metric in default_val_metric_dfs[next(iter(default_val_metric_dfs))]:
    for opt_cls in default_val_metric_dfs:
        plt.plot(default_val_metric_dfs[opt_cls][metric]['step'][:-1], default_val_metric_dfs[opt_cls][metric]['value'][:-1], label=opt_cls)
    plt.title(f'Default: Validation {metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    output_filename = f"default_run_plots/best_default_run_{metric}.png"
    plt.savefig(output_filename, dpi=300)
    print(f"\nPlot saved to {output_filename}")
    plt.show()


In [ ]:
results_df = pd.DataFrame(results).T
results_df = pd.concat([results_df, pd.DataFrame(optimizer_comparison_acc_df['Config']).set_index(results_df.index)], axis=1)
results_df

In [ ]:
results_df = pd.DataFrame(results).T
results_df = pd.concat([results_df, pd.DataFrame(optimizer_comparison_acc_df['Config']).set_index(results_df.index)], axis=1)

default_results_df = pd.DataFrame(default_results).T
default_results_df['Config'] = [{i: j for i, j in zip(inspect.getfullargspec(optimizer.__init__).args[2:], inspect.getfullargspec(optimizer.__init__).defaults)} for optimizer in OPTIMIZERS_PARAMS]

pd.concat([results_df, default_results_df])

In [ ]:
experiment_name = "tuned_regression_rs"
experiment_id = get_or_create_experiment(experiment_name)
mlflow.set_experiment(experiment_id=experiment_id)

rs_results = {}
random_seeds = np.random.randint(low=0, high=10**8, size=10)
for optimizer_type, config in zip(OPTIMIZERS_PARAMS.keys(), optimizer_comparison_acc_df["Config"]):
    rs_results[optimizer_type] = []
    for rs in random_seeds:
        seed_everything(rs)
        opt_name = str(optimizer_type)[str(optimizer_type).rfind('.')+1:-2]
        with mlflow.start_run(run_name=experiment_name+f"_opt_{opt_name}_rs_{rs}") as run:
            params_to_log = {i: j for i, j in config.items()}
            mlflow.log_params(params_to_log)
            
            mlflow.set_tag("optimizer", opt_name)
            mlflow.set_tag("random_seed", rs)
            model = SimpleRegressionModel(input_shape=INPUT_SHAPE, 
                                      hidden_units=HIDDEN_UNITS, 
                                      activation_function=nn.ReLU(),
                                     )
            loss_fn = nn.CrossEntropyLoss()
            wrapper = LightningWrapper(model=model, 
                                       optimizer_class=optimizer_type, 
                                       optimizer_hparams={i: j for i, j in config.items() if i not in ["experiment_name", "tracking_uri"]}, 
                                       loss_fn=loss_fn, )
            # mlflow.pytorch.autolog()
            trainer = pl.Trainer(#callbacks=[tune_callback, ],
                                limit_train_batches=BATCH_SIZE, 
                                max_epochs=NUM_EPOCHS,
                                accelerator="auto",
                                devices="auto",
                                enable_progress_bar=False,
                                #strategy="ddp_notebook",
                                enable_checkpointing=False,
                                #logger=False,
            )
            start = time.perf_counter()
            trainer.fit(model=wrapper, 
                        train_dataloaders=train_dataloader,
                        val_dataloaders=valid_dataloader,)
            end = time.perf_counter()
            time_training = end - start
            # final_metrics = trainer.callback_metrics
            metrics_to_save = trainer.test(model=wrapper, dataloaders=test_dataloader)[0]
            # print(metrics_to_save)
            metrics_to_save = {
                "test_mape": metrics_to_save.get("test_mape", 0.),
                "test_r2score": metrics_to_save.get("test_r2score", 0.),
            }
            
            metrics_to_save['time_training'] = time_training
            metrics_to_save['random_state'] = rs
            optimizer_name = str(optimizer_type)[str(optimizer_type).rfind('.')+1:-2]
            print(f"{optimizer_name} with random state {rs} was trained for {time_training} seconds")
            rs_results[optimizer_type].append(metrics_to_save)

            model.eval() 
    
            dummy_input = torch.randn(1, 3, 32, INPUT_SHAPE) 
        
            trial_name = session.get_trial_name()
        
            optimizer_name = str(optimizer_type).split('.')[-1].replace("'>", "")
        
            output_dir = "/kaggle/working/models/varying_random_state"
            os.makedirs(output_dir, exist_ok=True)
            
            onnx_file_path = os.path.join(output_dir, f"{optimizer_name}_{rs}.onnx")
            torch.onnx.export(
                model,
                dummy_input,
                onnx_file_path,
                export_params=True,
                opset_version=11,
                do_constant_folding=True,
                input_names=['input'],
                output_names=['output'],
                dynamic_axes={'input' : {0 : 'batch_size'},
                              'output' : {0 : 'batch_size'}}
            )
            print(f"Successfully exported ONNX model to: {onnx_file_path}")
            